# PENUNET_MR PLOT

In [2]:
# %% RELOAD bundle and re-plot (save PDF + PNG to same subfolder)
import os
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize

# ---- Fonts (Times-like) + avoid Type 3) ----
mpl.rcParams.update({
    'font.family': ['Times New Roman', 'STIXGeneral', 'DejaVu Serif'],
    'mathtext.fontset': 'stix',
    'text.usetex': False,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'svg.fonttype': 'none',
})

# ===== SETTINGS =====
PARENT   = "save_plots"
SUB      = "PENUNET_MR"
BUNDLE   = "PENUNET_MR_snap25.npz"   # change if you saved a different name
PNG_DPI  = 1080
# ====================

path = os.path.join(PARENT, SUB, BUNDLE)
if not os.path.exists(path):
    raise FileNotFoundError("Bundle not found: " + os.path.abspath(path))

Z = np.load(path, allow_pickle=True)

# unpack
SNAP   = int(Z["snap"])
x_ic   = Z["x_ic"];  y_ic = Z["y_ic"]
u_star = Z["u_star"]; v_star = Z["v_star"]
u_pred = Z["u_pred"]; v_pred = Z["v_pred"]
XLIM   = tuple(Z["xlim"])
CMAP   = str(Z["cmap"])

label_ux     = Z["label_ux"].item()
label_ux_hat = Z["label_ux_hat"].item()
label_uy     = Z["label_uy"].item()
label_uy_hat = Z["label_uy_hat"].item()

umin_u = float(Z["umin_u"]); umax_u = float(Z["umax_u"])
vmin_v = float(Z["vmin_v"]); vmax_v = float(Z["vmax_v"])

norm_u = Normalize(vmin=umin_u, vmax=umax_u)
norm_v = Normalize(vmin=vmin_v, vmax=vmax_v)

def plot_deformation(
    x, y, u, v, comp_label, norm, component='u',
    cmap='jet', xlim=None, textbox_xy=(0.035, 0.05),
    title=None, save_stub=None,
    # --- font controls ---
    cbar_label_fs=28, cbar_tick_fs=18, box_fs=21,
    label_fs=28, tick_fs=18, title_fs=20,
):
    x = x.ravel(); y = y.ravel()
    u = u.ravel(); v = v.ravel()

    # sign convention
    x_def = -x - u
    y_def =  y + v

    if component == 'u':
        cfield = u
        label_text = f"Max {comp_label}: {u.max():.2f}\nMin {comp_label}: {u.min():.2f}"
    else:
        cfield = v
        label_text = f"Max {comp_label}: {v.max():.2f}\nMin {comp_label}: {v.min():.2f}"

    fig, ax = plt.subplots(figsize=(10, 6), dpi=1080)
    ax.scatter(-x, y, color='grey', alpha=1, s=0.009)                       # undeformed
    ax.scatter(x_def, y_def, c=cfield, cmap=cmap, norm=norm, s=1, alpha=1)  # deformed

    # Colorbar with font-size control
    cbar = plt.colorbar(plt.cm.ScalarMappable(cmap=cmap, norm=norm), ax=ax)
    # cbar.set_label('Deformation', fontsize=cbar_label_fs)
    cbar.ax.tick_params(labelsize=cbar_tick_fs)

    # Stats box (Max/Min) font size
    ax.text(textbox_xy[0], textbox_xy[1], label_text, transform=ax.transAxes,
            fontsize=box_fs, va='bottom',
            bbox=dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.5))

    # Axes labels / ticks / title
    ax.set_xlabel('X Coordinate', fontsize=label_fs)
    ax.set_ylabel('Y Coordinate', fontsize=label_fs)
    ax.tick_params(axis='both', labelsize=tick_fs)
    ax.set_aspect('equal', adjustable='box')

    if xlim is not None:
        ax.set_xlim(*xlim)
    if title:
        ax.set_title(title, fontsize=title_fs)

    fig.tight_layout()

    if save_stub:
        base = os.path.join(PARENT, SUB, save_stub)
        plt.savefig(base + ".pdf", bbox_inches='tight')
        plt.savefig(base + ".png", dpi=PNG_DPI, bbox_inches='tight')
        # print("Saved:", os.path.abspath(base + ".pdf"))
        # print("Saved:", os.path.abspath(base + ".png"))

    # plt.show()
    plt.close(fig)

# Reproduce the two plots (u_y GT and Prediction). Add u_x if needed.

plot_deformation(x_ic, y_ic, u_star, v_star, comp_label=label_ux, norm=norm_u, component='u',
                 cmap=CMAP, xlim=XLIM, title=rf"PENUNET-MR-EXACT u-X at snaP {SNAP}", save_stub=f"PENUNET_MR_ux_EXACT_snap{SNAP}_reload")
plot_deformation(x_ic, y_ic, u_pred, v_pred, comp_label=label_ux_hat, norm=norm_u, component='u',
                 cmap=CMAP, xlim=XLIM, title=rf"PENUNET-MR-Prediction u-x at snaP {SNAP}", save_stub=f"PENUNET_MR_ux_PRED_snap{SNAP}_reload")

plot_deformation(
    x_ic, y_ic, u_star, v_star,
    comp_label=label_uy, norm=norm_v, component='v', cmap=CMAP,
    xlim=XLIM, title=rf"PENUNET-MR-EXACT u-Y at snaP {SNAP}",
    save_stub=f"PENUNET_MR_uy_EXACT_snap{SNAP}_reload",
    # font sizes (tweak here as you like)
    cbar_label_fs=30, cbar_tick_fs=20, box_fs=22,
    label_fs=28, tick_fs=18, title_fs=22)

plot_deformation(x_ic, y_ic, u_pred, v_pred, comp_label=label_uy_hat, norm=norm_v,
                 component='v', cmap=CMAP, xlim=XLIM, title=rf"PENUNET-MR-Prediction u-Y at snaP {SNAP}",
                 save_stub=f"PENUNET_MR_uy_PRED_snap{SNAP}_reload",
                 cbar_label_fs=30, cbar_tick_fs=20, box_fs=22, label_fs=28, tick_fs=18, title_fs=22)



# PENUNET_YEOH PLOT

In [1]:
# %% RELOAD bundle and re-plot (save PDF + PNG to same subfolder)
import os
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize

# ---- Fonts (Times-like) + avoid Type 3) ----
mpl.rcParams.update({
    'font.family': ['Times New Roman', 'STIXGeneral', 'DejaVu Serif'],
    'mathtext.fontset': 'stix',
    'text.usetex': False,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'svg.fonttype': 'none',
})

# ===== SETTINGS =====
PARENT   = "save_plots"
SUB      = "PENUNET_YEOH"
BUNDLE   = "PENUNET_YEOH_snap25.npz"   # change if you saved a different name
PNG_DPI  = 1080
# ====================

path = os.path.join(PARENT, SUB, BUNDLE)
if not os.path.exists(path):
    raise FileNotFoundError("Bundle not found: " + os.path.abspath(path))

Z = np.load(path, allow_pickle=True)

# unpack
SNAP   = int(Z["snap"])
x_ic   = Z["x_ic"];  y_ic = Z["y_ic"]
u_star = Z["u_star"]; v_star = Z["v_star"]
u_pred = Z["u_pred"]; v_pred = Z["v_pred"]
XLIM   = tuple(Z["xlim"])
CMAP   = str(Z["cmap"])

label_ux     = Z["label_ux"].item()
label_ux_hat = Z["label_ux_hat"].item()
label_uy     = Z["label_uy"].item()
label_uy_hat = Z["label_uy_hat"].item()

umin_u = float(Z["umin_u"]); umax_u = float(Z["umax_u"])
vmin_v = float(Z["vmin_v"]); vmax_v = float(Z["vmax_v"])

norm_u = Normalize(vmin=umin_u, vmax=umax_u)
norm_v = Normalize(vmin=vmin_v, vmax=vmax_v)

def plot_deformation(
    x, y, u, v, comp_label, norm, component='u',
    cmap='jet', xlim=None, textbox_xy=(0.035, 0.05),
    title=None, save_stub=None,
    # --- font controls ---
    cbar_label_fs=28, cbar_tick_fs=18, box_fs=21,
    label_fs=28, tick_fs=18, title_fs=20,
):
    x = x.ravel(); y = y.ravel()
    u = u.ravel(); v = v.ravel()

    # sign convention
    x_def = -x - u
    y_def =  y + v

    if component == 'u':
        cfield = u
        label_text = f"Max {comp_label}: {u.max():.2f}\nMin {comp_label}: {u.min():.2f}"
    else:
        cfield = v
        label_text = f"Max {comp_label}: {v.max():.2f}\nMin {comp_label}: {v.min():.2f}"

    fig, ax = plt.subplots(figsize=(10, 6), dpi=1080)
    ax.scatter(-x, y, color='grey', alpha=1, s=0.009)                       # undeformed
    ax.scatter(x_def, y_def, c=cfield, cmap=cmap, norm=norm, s=1, alpha=1)  # deformed

    # Colorbar with font-size control
    cbar = plt.colorbar(plt.cm.ScalarMappable(cmap=cmap, norm=norm), ax=ax)
    # cbar.set_label('Deformation', fontsize=cbar_label_fs)
    cbar.ax.tick_params(labelsize=cbar_tick_fs)

    # Stats box (Max/Min) font size
    ax.text(textbox_xy[0], textbox_xy[1], label_text, transform=ax.transAxes,
            fontsize=box_fs, va='bottom',
            bbox=dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.5))

    # Axes labels / ticks / title
    ax.set_xlabel('X Coordinate', fontsize=label_fs)
    ax.set_ylabel('Y Coordinate', fontsize=label_fs)
    ax.tick_params(axis='both', labelsize=tick_fs)
    ax.set_aspect('equal', adjustable='box')

    if xlim is not None:
        ax.set_xlim(*xlim)
    if title:
        ax.set_title(title, fontsize=title_fs)

    fig.tight_layout()

    if save_stub:
        base = os.path.join(PARENT, SUB, save_stub)
        plt.savefig(base + ".pdf", bbox_inches='tight')
        plt.savefig(base + ".png", dpi=PNG_DPI, bbox_inches='tight')
        # print("Saved:", os.path.abspath(base + ".pdf"))
        # print("Saved:", os.path.abspath(base + ".png"))

    # plt.show()
    plt.close(fig)

# Reproduce the two plots (u_y GT and Prediction). Add u_x if needed.

plot_deformation(x_ic, y_ic, u_star, v_star, comp_label=label_ux, norm=norm_u, component='u',
                 cmap=CMAP, xlim=XLIM, title=rf"PENUNET-YEOH-EXACT u-X at snap {SNAP}", save_stub=f"PENUNET_YEOH_ux_EXACT_snap{SNAP}_reload")
plot_deformation(x_ic, y_ic, u_pred, v_pred, comp_label=label_ux_hat, norm=norm_u, component='u',
                 cmap=CMAP, xlim=XLIM, title=rf"PENUNET-YEOH-Prediction u-x at snap {SNAP}", save_stub=f"PENUNET_YEOH_ux_PRED_snap{SNAP}_reload")

plot_deformation(
    x_ic, y_ic, u_star, v_star,
    comp_label=label_uy, norm=norm_v, component='v', cmap=CMAP,
    xlim=XLIM, title=rf"PENUNET-YEOH-EXACT u-Y at snap {SNAP}",
    save_stub=f"PENUNET_YEOH_uy_EXACT_snap{SNAP}_reload",
    # font sizes (tweak here as you like)
    cbar_label_fs=30, cbar_tick_fs=20, box_fs=22,
    label_fs=28, tick_fs=18, title_fs=22)

plot_deformation(x_ic, y_ic, u_pred, v_pred, comp_label=label_uy_hat, norm=norm_v,
                 component='v', cmap=CMAP, xlim=XLIM, title=rf"PENUNET-YEOH-Prediction u-Y at snap {SNAP}",
                 save_stub=f"PENUNET_YEOH_uy_PRED_snap{SNAP}_reload",
                 cbar_label_fs=30, cbar_tick_fs=20, box_fs=22, label_fs=28, tick_fs=18, title_fs=22)



# BELLOW_MR PLOT

In [1]:
# %% RELOAD bundle and re-plot (save PDF + PNG to same subfolder)
import os
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize

# ---- Fonts (Times-like) + avoid Type 3) ----
mpl.rcParams.update({
    'font.family': ['Times New Roman', 'STIXGeneral', 'DejaVu Serif'],
    'mathtext.fontset': 'stix',
    'text.usetex': False,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'svg.fonttype': 'none',
})

# ===== SETTINGS =====
PARENT   = "save_plots"
SUB      = "BELLOW_MR"
BUNDLE   = "BELLOW_MR_snap30.npz"   # change if you saved a different name
PNG_DPI  = 1080
# ====================

path = os.path.join(PARENT, SUB, BUNDLE)
if not os.path.exists(path):
    raise FileNotFoundError("Bundle not found: " + os.path.abspath(path))

Z = np.load(path, allow_pickle=True)

# unpack
SNAP   = int(Z["snap"])
x_ic   = Z["x_ic"];  y_ic = Z["y_ic"]
u_star = Z["u_star"]; v_star = Z["v_star"]
u_pred = Z["u_pred"]; v_pred = Z["v_pred"]
XLIM   = tuple(Z["xlim"])
CMAP   = str(Z["cmap"])

label_ux     = Z["label_ux"].item()
label_ux_hat = Z["label_ux_hat"].item()
label_uy     = Z["label_uy"].item()
label_uy_hat = Z["label_uy_hat"].item()

umin_u = float(Z["umin_u"]); umax_u = float(Z["umax_u"])
vmin_v = float(Z["vmin_v"]); vmax_v = float(Z["vmax_v"])

norm_u = Normalize(vmin=umin_u, vmax=umax_u)
norm_v = Normalize(vmin=vmin_v, vmax=vmax_v)

def plot_deformation(
    x, y, u, v, comp_label, norm, component='u',
    cmap='jet', xlim=None, textbox_xy=(0.035, 0.05),
    title=None, save_stub=None,
    # --- font controls ---
    cbar_label_fs=28, cbar_tick_fs=18, box_fs=21,
    label_fs=28, tick_fs=18, title_fs=20,
):
    x = x.ravel(); y = y.ravel()
    u = u.ravel(); v = v.ravel()

    # sign convention
    x_def = -x - u
    y_def =  y + v

    if component == 'u':
        cfield = u
        label_text = f"Max {comp_label}: {u.max():.2f}\nMin {comp_label}: {u.min():.2f}"
    else:
        cfield = v
        label_text = f"Max {comp_label}: {v.max():.2f}\nMin {comp_label}: {v.min():.2f}"

    fig, ax = plt.subplots(figsize=(10, 6), dpi=1080)
    ax.scatter(-x, y, color='grey', alpha=1, s=0.009)                       # undeformed
    ax.scatter(x_def, y_def, c=cfield, cmap=cmap, norm=norm, s=1, alpha=1)  # deformed

    # Colorbar with font-size control
    cbar = plt.colorbar(plt.cm.ScalarMappable(cmap=cmap, norm=norm), ax=ax)
    # cbar.set_label('Deformation', fontsize=cbar_label_fs)
    cbar.ax.tick_params(labelsize=cbar_tick_fs)

    # Stats box (Max/Min) font size
    ax.text(textbox_xy[0], textbox_xy[1], label_text, transform=ax.transAxes,
            fontsize=box_fs, va='bottom',
            bbox=dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.5))

    # Axes labels / ticks / title
    ax.set_xlabel('X Coordinate', fontsize=label_fs)
    ax.set_ylabel('Y Coordinate', fontsize=label_fs)
    ax.tick_params(axis='both', labelsize=tick_fs)
    ax.set_aspect('equal', adjustable='box')

    if xlim is not None:
        ax.set_xlim(*xlim)
    if title:
        ax.set_title(title, fontsize=title_fs)

    fig.tight_layout()

    if save_stub:
        base = os.path.join(PARENT, SUB, save_stub)
        plt.savefig(base + ".pdf", bbox_inches='tight')
        plt.savefig(base + ".png", dpi=PNG_DPI, bbox_inches='tight')
    #     print("Saved:", os.path.abspath(base + ".pdf"))
    #     print("Saved:", os.path.abspath(base + ".png"))

    # plt.show()
    plt.close(fig)

# Reproduce the two plots (u_y GT and Prediction). Add u_x if needed.

plot_deformation(x_ic, y_ic, u_star, v_star, comp_label=label_ux, norm=norm_u, component='u',
                 cmap=CMAP, xlim=XLIM, title=rf"BELLOW-MR-EXACT u-X at snap {SNAP}", save_stub=f"BELLOW_MR_ux_EXACT_snap{SNAP}_reload")
plot_deformation(x_ic, y_ic, u_pred, v_pred, comp_label=label_ux_hat, norm=norm_u, component='u',
                 cmap=CMAP, xlim=XLIM, title=rf"BELLOW-MR-Prediction u-x at snap {SNAP}", save_stub=f"BELLOW_MR_ux_PRED_snap{SNAP}_reload")

plot_deformation(
    x_ic, y_ic, u_star, v_star,
    comp_label=label_uy, norm=norm_v, component='v', cmap=CMAP,
    xlim=XLIM, title=rf"BELLOW-MR-EXACT u-Y at snap {SNAP}",
    save_stub=f"BELLOW_MR_uy_EXACT_snap{SNAP}_reload",
    # font sizes (tweak here as you like)
    cbar_label_fs=30, cbar_tick_fs=20, box_fs=22,
    label_fs=28, tick_fs=18, title_fs=22)

plot_deformation(x_ic, y_ic, u_pred, v_pred, comp_label=label_uy_hat, norm=norm_v,
                 component='v', cmap=CMAP, xlim=XLIM, title=rf"BELLOW-MR-Prediction u-Y at snap {SNAP}",
                 save_stub=f"BELLOW_MR_uy_PRED_snap{SNAP}_reload",
                 cbar_label_fs=30, cbar_tick_fs=20, box_fs=22, label_fs=28, tick_fs=18, title_fs=22)



# BELLOW_YEOH PLOT

In [2]:
# %% RELOAD bundle and re-plot (save PDF + PNG to same subfolder)
import os
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize

# ---- Fonts (Times-like) + avoid Type 3) ----
mpl.rcParams.update({
    'font.family': ['Times New Roman', 'STIXGeneral', 'DejaVu Serif'],
    'mathtext.fontset': 'stix',
    'text.usetex': False,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'svg.fonttype': 'none',
})

# ===== SETTINGS =====
PARENT   = "save_plots"
SUB      = "BELLOW_YEOH"
BUNDLE   = "BELLOW_YEOH_snap30.npz"   # change if you saved a different name
PNG_DPI  = 1080
# ====================

path = os.path.join(PARENT, SUB, BUNDLE)
if not os.path.exists(path):
    raise FileNotFoundError("Bundle not found: " + os.path.abspath(path))

Z = np.load(path, allow_pickle=True)

# unpack
SNAP   = int(Z["snap"])
x_ic   = Z["x_ic"];  y_ic = Z["y_ic"]
u_star = Z["u_star"]; v_star = Z["v_star"]
u_pred = Z["u_pred"]; v_pred = Z["v_pred"]
XLIM   = tuple(Z["xlim"])
CMAP   = str(Z["cmap"])

label_ux     = Z["label_ux"].item()
label_ux_hat = Z["label_ux_hat"].item()
label_uy     = Z["label_uy"].item()
label_uy_hat = Z["label_uy_hat"].item()

umin_u = float(Z["umin_u"]); umax_u = float(Z["umax_u"])
vmin_v = float(Z["vmin_v"]); vmax_v = float(Z["vmax_v"])

norm_u = Normalize(vmin=umin_u, vmax=umax_u)
norm_v = Normalize(vmin=vmin_v, vmax=vmax_v)

def plot_deformation(
    x, y, u, v, comp_label, norm, component='u',
    cmap='jet', xlim=None, textbox_xy=(0.035, 0.05),
    title=None, save_stub=None,
    # --- font controls ---
    cbar_label_fs=28, cbar_tick_fs=18, box_fs=21,
    label_fs=28, tick_fs=18, title_fs=20,
):
    x = x.ravel(); y = y.ravel()
    u = u.ravel(); v = v.ravel()

    # sign convention
    x_def = -x - u
    y_def =  y + v

    if component == 'u':
        cfield = u
        label_text = f"Max {comp_label}: {u.max():.2f}\nMin {comp_label}: {u.min():.2f}"
    else:
        cfield = v
        label_text = f"Max {comp_label}: {v.max():.2f}\nMin {comp_label}: {v.min():.2f}"

    fig, ax = plt.subplots(figsize=(10, 6), dpi=1080)
    ax.scatter(-x, y, color='grey', alpha=1, s=0.009)                       # undeformed
    ax.scatter(x_def, y_def, c=cfield, cmap=cmap, norm=norm, s=1, alpha=1)  # deformed

    # Colorbar with font-size control
    cbar = plt.colorbar(plt.cm.ScalarMappable(cmap=cmap, norm=norm), ax=ax)
    # cbar.set_label('Deformation', fontsize=cbar_label_fs)
    cbar.ax.tick_params(labelsize=cbar_tick_fs)

    # Stats box (Max/Min) font size
    ax.text(textbox_xy[0], textbox_xy[1], label_text, transform=ax.transAxes,
            fontsize=box_fs, va='bottom',
            bbox=dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.5))

    # Axes labels / ticks / title
    ax.set_xlabel('X Coordinate', fontsize=label_fs)
    ax.set_ylabel('Y Coordinate', fontsize=label_fs)
    ax.tick_params(axis='both', labelsize=tick_fs)
    ax.set_aspect('equal', adjustable='box')

    if xlim is not None:
        ax.set_xlim(*xlim)
    if title:
        ax.set_title(title, fontsize=title_fs)

    fig.tight_layout()

    if save_stub:
        base = os.path.join(PARENT, SUB, save_stub)
        plt.savefig(base + ".pdf", bbox_inches='tight')
        plt.savefig(base + ".png", dpi=PNG_DPI, bbox_inches='tight')
    #     print("Saved:", os.path.abspath(base + ".pdf"))
    #     print("Saved:", os.path.abspath(base + ".png"))

    # plt.show()
    plt.close(fig)

# Reproduce the two plots (u_y GT and Prediction). Add u_x if needed.

plot_deformation(x_ic, y_ic, u_star, v_star, comp_label=label_ux, norm=norm_u, component='u',
                 cmap=CMAP, xlim=XLIM, title=rf"BELLOW-YEOH-EXACT u-X at snap {SNAP}", save_stub=f"BELLOW_YEOH_ux_EXACT_snap{SNAP}_reload")
plot_deformation(x_ic, y_ic, u_pred, v_pred, comp_label=label_ux_hat, norm=norm_u, component='u',
                 cmap=CMAP, xlim=XLIM, title=rf"BELLOW-YEOH-Prediction u-x at snap {SNAP}", save_stub=f"BELLOW_YEOH_ux_PRED_snap{SNAP}_reload")

plot_deformation(
    x_ic, y_ic, u_star, v_star,
    comp_label=label_uy, norm=norm_v, component='v', cmap=CMAP,
    xlim=XLIM, title=rf"BELLOW-YEOH-EXACT u-Y at snap {SNAP}",
    save_stub=f"BELLOW_YEOH_uy_EXACT_snap{SNAP}_reload",
    # font sizes (tweak here as you like)
    cbar_label_fs=30, cbar_tick_fs=20, box_fs=22,
    label_fs=28, tick_fs=18, title_fs=22)

plot_deformation(x_ic, y_ic, u_pred, v_pred, comp_label=label_uy_hat, norm=norm_v,
                 component='v', cmap=CMAP, xlim=XLIM, title=rf"BELLOW-YEOH-Prediction u-Y at snap {SNAP}",
                 save_stub=f"BELLOW_YEOH_uy_PRED_snap{SNAP}_reload",
                 cbar_label_fs=30, cbar_tick_fs=20, box_fs=22, label_fs=28, tick_fs=18, title_fs=22)



# BENDING_JOINT_MR PLOT

In [1]:
# %% RELOAD bundle and re-plot (save PDF + PNG to same subfolder)
import os
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize

# ---- Fonts (Times-like) + avoid Type 3) ----
mpl.rcParams.update({
    'font.family': ['Times New Roman', 'STIXGeneral', 'DejaVu Serif'],
    'mathtext.fontset': 'stix',
    'text.usetex': False,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'svg.fonttype': 'none',
})

# ===== SETTINGS =====
PARENT   = "save_plots"
SUB      = "BENDING_JOINT_MR"
BUNDLE   = "BENDING_JOINT_MR_snap45.npz"   # change if you saved a different name
PNG_DPI  = 1080
# ====================

path = os.path.join(PARENT, SUB, BUNDLE)
if not os.path.exists(path):
    raise FileNotFoundError("Bundle not found: " + os.path.abspath(path))

Z = np.load(path, allow_pickle=True)

# unpack
SNAP   = int(Z["snap"])
x_ic   = Z["x_ic"];  y_ic = Z["y_ic"]
u_star = Z["u_star"]; v_star = Z["v_star"]
u_pred = Z["u_pred"]; v_pred = Z["v_pred"]
XLIM   = tuple(Z["xlim"])
CMAP   = str(Z["cmap"])

label_ux     = Z["label_ux"].item()
label_ux_hat = Z["label_ux_hat"].item()
label_uy     = Z["label_uy"].item()
label_uy_hat = Z["label_uy_hat"].item()

umin_u = float(Z["umin_u"]); umax_u = float(Z["umax_u"])
vmin_v = float(Z["vmin_v"]); vmax_v = float(Z["vmax_v"])

norm_u = Normalize(vmin=umin_u, vmax=umax_u)
norm_v = Normalize(vmin=vmin_v, vmax=vmax_v)

def plot_deformation(
    x, y, u, v, comp_label, norm, component='u',
    cmap='jet', xlim=None, textbox_xy=(0.035, 0.05),
    title=None, save_stub=None,
    # --- font controls ---
    cbar_label_fs=28, cbar_tick_fs=18, box_fs=21,
    label_fs=28, tick_fs=18, title_fs=20,
):
    x = x.ravel(); y = y.ravel()
    u = u.ravel(); v = v.ravel()

    # sign convention
    x_def = -x - u
    y_def =  y + v

    if component == 'u':
        cfield = u
        label_text = f"Max {comp_label}: {u.max():.2f}\nMin {comp_label}: {u.min():.2f}"
    else:
        cfield = v
        label_text = f"Max {comp_label}: {v.max():.2f}\nMin {comp_label}: {v.min():.2f}"

    fig, ax = plt.subplots(figsize=(10, 6), dpi=1080)
    ax.scatter(-x, y, color='grey', alpha=1, s=0.009)                       # undeformed
    ax.scatter(x_def, y_def, c=cfield, cmap=cmap, norm=norm, s=1, alpha=1)  # deformed

    # Colorbar with font-size control
    cbar = plt.colorbar(plt.cm.ScalarMappable(cmap=cmap, norm=norm), ax=ax)
    # cbar.set_label('Deformation', fontsize=cbar_label_fs)
    cbar.ax.tick_params(labelsize=cbar_tick_fs)

    # Stats box (Max/Min) font size
    ax.text(textbox_xy[0], textbox_xy[1], label_text, transform=ax.transAxes,
            fontsize=box_fs, va='bottom',
            bbox=dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.5))

    # Axes labels / ticks / title
    ax.set_xlabel('X Coordinate', fontsize=label_fs)
    ax.set_ylabel('Y Coordinate', fontsize=label_fs)
    ax.tick_params(axis='both', labelsize=tick_fs)
    ax.set_aspect('equal', adjustable='box')

    if xlim is not None:
        ax.set_xlim(*xlim)
    if title:
        ax.set_title(title, fontsize=title_fs)

    fig.tight_layout()

    if save_stub:
        base = os.path.join(PARENT, SUB, save_stub)
        plt.savefig(base + ".pdf", bbox_inches='tight')
        plt.savefig(base + ".png", dpi=PNG_DPI, bbox_inches='tight')
    #     print("Saved:", os.path.abspath(base + ".pdf"))
    #     print("Saved:", os.path.abspath(base + ".png"))

    # plt.show()
    plt.close(fig)

# Reproduce the two plots (u_y GT and Prediction). Add u_x if needed.

plot_deformation(x_ic, y_ic, u_star, v_star, comp_label=label_ux, norm=norm_u, component='u',
                 cmap=CMAP, xlim=XLIM, title=rf"BENDING-JOINT-MR-EXACT u-X at snap {SNAP}", save_stub=f"BENDING_JOINT_MR_ux_EXACT_snap{SNAP}_reload")
plot_deformation(x_ic, y_ic, u_pred, v_pred, comp_label=label_ux_hat, norm=norm_u, component='u',
                 cmap=CMAP, xlim=XLIM, title=rf"BENDING-JOINT-MR-Prediction u-x at snap {SNAP}", save_stub=f"BENDING_JOINT_MR_ux_PRED_snap{SNAP}_reload")

plot_deformation(
    x_ic, y_ic, u_star, v_star,
    comp_label=label_uy, norm=norm_v, component='v', cmap=CMAP,
    xlim=XLIM, title=rf"BENDING-JOINT-MR-EXACT u-Y at snap {SNAP}",
    save_stub=f"BENDING_JOINT_MR_uy_EXACT_snap{SNAP}_reload",
    # font sizes (tweak here as you like)
    cbar_label_fs=30, cbar_tick_fs=20, box_fs=22,
    label_fs=28, tick_fs=18, title_fs=22)

plot_deformation(x_ic, y_ic, u_pred, v_pred, comp_label=label_uy_hat, norm=norm_v,
                 component='v', cmap=CMAP, xlim=XLIM, title=rf"BENDING-JOINT-MR-Prediction u-Y at snap {SNAP}",
                 save_stub=f"BENDING_JOINT_MR_uy_PRED_snap{SNAP}_reload",
                 cbar_label_fs=30, cbar_tick_fs=20, box_fs=22, label_fs=28, tick_fs=18, title_fs=22)



# BENDING_JOINT_YEOH PLOT

In [1]:
# %% RELOAD bundle and re-plot (save PDF + PNG to same subfolder)
import os
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize

# ---- Fonts (Times-like) + avoid Type 3) ----
mpl.rcParams.update({
    'font.family': ['Times New Roman', 'STIXGeneral', 'DejaVu Serif'],
    'mathtext.fontset': 'stix',
    'text.usetex': False,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'svg.fonttype': 'none',
})

# ===== SETTINGS =====
PARENT   = "save_plots"
SUB      = "BENDING_JOINT_YEOH"
BUNDLE   = "BENDING_JOINT_YEOH_snap45.npz"   # change if you saved a different name
PNG_DPI  = 1080
# ====================

path = os.path.join(PARENT, SUB, BUNDLE)
if not os.path.exists(path):
    raise FileNotFoundError("Bundle not found: " + os.path.abspath(path))

Z = np.load(path, allow_pickle=True)

# unpack
SNAP   = int(Z["snap"])
x_ic   = Z["x_ic"];  y_ic = Z["y_ic"]
u_star = Z["u_star"]; v_star = Z["v_star"]
u_pred = Z["u_pred"]; v_pred = Z["v_pred"]
XLIM   = tuple(Z["xlim"])
CMAP   = str(Z["cmap"])

label_ux     = Z["label_ux"].item()
label_ux_hat = Z["label_ux_hat"].item()
label_uy     = Z["label_uy"].item()
label_uy_hat = Z["label_uy_hat"].item()

umin_u = float(Z["umin_u"]); umax_u = float(Z["umax_u"])
vmin_v = float(Z["vmin_v"]); vmax_v = float(Z["vmax_v"])

norm_u = Normalize(vmin=umin_u, vmax=umax_u)
norm_v = Normalize(vmin=vmin_v, vmax=vmax_v)

def plot_deformation(
    x, y, u, v, comp_label, norm, component='u',
    cmap='jet', xlim=None, textbox_xy=(0.035, 0.05),
    title=None, save_stub=None,
    # --- font controls ---
    cbar_label_fs=28, cbar_tick_fs=18, box_fs=21,
    label_fs=28, tick_fs=18, title_fs=20,
):
    x = x.ravel(); y = y.ravel()
    u = u.ravel(); v = v.ravel()

    # sign convention
    x_def = -x - u
    y_def =  y + v

    if component == 'u':
        cfield = u
        label_text = f"Max {comp_label}: {u.max():.2f}\nMin {comp_label}: {u.min():.2f}"
    else:
        cfield = v
        label_text = f"Max {comp_label}: {v.max():.2f}\nMin {comp_label}: {v.min():.2f}"

    fig, ax = plt.subplots(figsize=(10, 6), dpi=1080)
    ax.scatter(-x, y, color='grey', alpha=1, s=0.009)                       # undeformed
    ax.scatter(x_def, y_def, c=cfield, cmap=cmap, norm=norm, s=1, alpha=1)  # deformed

    # Colorbar with font-size control
    cbar = plt.colorbar(plt.cm.ScalarMappable(cmap=cmap, norm=norm), ax=ax)
    # cbar.set_label('Deformation', fontsize=cbar_label_fs)
    cbar.ax.tick_params(labelsize=cbar_tick_fs)

    # Stats box (Max/Min) font size
    ax.text(textbox_xy[0], textbox_xy[1], label_text, transform=ax.transAxes,
            fontsize=box_fs, va='bottom',
            bbox=dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.5))

    # Axes labels / ticks / title
    ax.set_xlabel('X Coordinate', fontsize=label_fs)
    ax.set_ylabel('Y Coordinate', fontsize=label_fs)
    ax.tick_params(axis='both', labelsize=tick_fs)
    ax.set_aspect('equal', adjustable='box')

    if xlim is not None:
        ax.set_xlim(*xlim)
    if title:
        ax.set_title(title, fontsize=title_fs)

    fig.tight_layout()

    if save_stub:
        base = os.path.join(PARENT, SUB, save_stub)
        plt.savefig(base + ".pdf", bbox_inches='tight')
        plt.savefig(base + ".png", dpi=PNG_DPI, bbox_inches='tight')
    #     print("Saved:", os.path.abspath(base + ".pdf"))
    #     print("Saved:", os.path.abspath(base + ".png"))

    # plt.show()
    plt.close(fig)

# Reproduce the two plots (u_y GT and Prediction). Add u_x if needed.

plot_deformation(x_ic, y_ic, u_star, v_star, comp_label=label_ux, norm=norm_u, component='u',
                 cmap=CMAP, xlim=XLIM, title=rf"BENDING-JOINT-YEOH-EXACT u-X at snap {SNAP}", save_stub=f"BENDING_JOINT_YEOH_ux_EXACT_snap{SNAP}_reload")
plot_deformation(x_ic, y_ic, u_pred, v_pred, comp_label=label_ux_hat, norm=norm_u, component='u',
                 cmap=CMAP, xlim=XLIM, title=rf"BENDING-JOINT-YEOH-Prediction u-x at snap {SNAP}", save_stub=f"BENDING_JOINT_YEOH_ux_PRED_snap{SNAP}_reload")

plot_deformation(
    x_ic, y_ic, u_star, v_star,
    comp_label=label_uy, norm=norm_v, component='v', cmap=CMAP,
    xlim=XLIM, title=rf"BENDING-JOINT-YEOH-EXACT u-Y at snap {SNAP}",
    save_stub=f"BENDING_JOINT_YEOH_uy_EXACT_snap{SNAP}_reload",
    # font sizes (tweak here as you like)
    cbar_label_fs=30, cbar_tick_fs=20, box_fs=22,
    label_fs=28, tick_fs=18, title_fs=22)

plot_deformation(x_ic, y_ic, u_pred, v_pred, comp_label=label_uy_hat, norm=norm_v,
                 component='v', cmap=CMAP, xlim=XLIM, title=rf"BENDING-JOINT-YEOH-Prediction u-Y at snap {SNAP}",
                 save_stub=f"BENDING_JOINT_YEOH_uy_PRED_snap{SNAP}_reload",
                 cbar_label_fs=30, cbar_tick_fs=20, box_fs=22, label_fs=28, tick_fs=18, title_fs=22)

